# Βήμα 6 – Σύστημα Α με μειωμένα χαρακτηριστικά
Στο notebook 08 είδαμε ότι τα 39 αριθμητικά χαρακτηριστικά του UNSW-NB15 έχουν μεγάλη ετερο-συσχέτιση και ότι με **8** από αυτά
(κανένα ζεύγος με |r| > 0.5) το Random Forest χάνει ελάχιστα. Εδώ ελέγχουμε αν ισχύει το ίδιο για το **νευρωνικό δίκτυο**.

1. Ξαναφτιάχνουμε τα δεδομένα με τον ίδιο ακριβώς τρόπο (ίδιες γραμμές σε train / validation / test).
2. Ελέγχουμε αν τα **κατηγορικά** (`proto`, `service`, `state`) ξαναφέρνουν πλεονασμό.
3. Εκπαιδεύουμε το ίδιο δίκτυο σε 4 εκδοχές εισόδων και συγκρίνουμε με το αρχικό (39 + κατηγορικά).

**Η επιλογή γίνεται στο validation**· το test μόνο για την τελική αναφορά. Χρόνος: περίπου 15–25 λεπτά συνολικά.

In [ ]:
import json
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import keras
from keras import layers
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch
from scipy.stats import chi2_contingency
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, classification_report)

DATA, MODELS, FIG = "../data/", "../models/", "../figures/"
SEED = 42
POWER = 0.25               # βάρη κλάσεων: η επιλογή του notebook 07

with open(MODELS + "unsw_meta.json", encoding="utf-8") as f:
    meta = json.load(f)
ATTACK_CLASSES, SEVERITY_CLASSES = meta["attack_classes"], meta["severity_classes"]
SEV_OF_ATTACK = np.array([SEVERITY_CLASSES.index(meta["severity_map"][c]) for c in ATTACK_CLASSES])
CAT_COLS, NUM_COLS, LOG_COLS = meta["cat_cols"], meta["num_cols"], meta["log_cols"]

# Οι λίστες βγήκαν στο notebook 08 (κατώφλι συσχέτισης 0.5 και 0.7)
FEATS_8 = ["dur", "dpkts", "tcprtt", "smean", "trans_depth", "ct_srv_src", "is_ftp_login", "is_sm_ips_ports"]
FEATS_13 = ["dur", "spkts", "rate", "sttl", "stcpb", "dtcpb", "tcprtt", "smean", "trans_depth",
            "ct_srv_src", "ct_state_ttl", "is_ftp_login", "is_sm_ips_ports"]

## 6.1 Δεδομένα – ίδια βήματα με το notebook 03
Ίδιος καθαρισμός, ίδια αφαίρεση διπλοτύπων, ίδιος σπόρος στον διαχωρισμό → **οι ίδιες γραμμές** σε train και validation.
Στο τέλος το ελέγχουμε συγκρίνοντας με το αποθηκευμένο `unsw_prepared.npz`.

In [ ]:
train_full = pd.read_csv(DATA + "UNSW_NB15_train-set.csv")
test_df = pd.read_csv(DATA + "UNSW_NB15_test-set.csv")
for df in (train_full, test_df):
    df["attack_cat"] = df["attack_cat"].str.strip()
    df["severity"] = df["attack_cat"].map(meta["severity_map"])
if meta["drop_duplicates"]:
    train_full = train_full.drop_duplicates(subset=[c for c in train_full.columns if c != "id"])

train_df, val_df = train_test_split(train_full, test_size=0.15, stratify=train_full["attack_cat"],
                                    random_state=meta["random_state"])
frames = {"train": train_df, "val": val_df, "test": test_df}

def encode_targets(df):
    return {"label": df["label"].to_numpy("int64"),
            "attack_cat": df["attack_cat"].map({c: i for i, c in enumerate(ATTACK_CLASSES)}).to_numpy("int64"),
            "severity": df["severity"].map({c: i for i, c in enumerate(SEVERITY_CLASSES)}).to_numpy("int64")}
y = {s: encode_targets(df) for s, df in frames.items()}

def targets(s):
    return [y[s]["label"].astype("float32").reshape(-1, 1), y[s]["attack_cat"].astype("int32"), y[s]["severity"].astype("int32")]

def make_preprocessor(num_cols, use_cat):
    """Ίδιος μετασχηματισμός με το notebook 03, αλλά μόνο για τις στήλες που ζητάμε."""
    log_cols = [c for c in LOG_COLS if c in num_cols]
    plain_cols = [c for c in num_cols if c not in LOG_COLS]
    steps = []
    if log_cols:
        steps.append(("log", Pipeline([("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
                                       ("scale", StandardScaler())]), log_cols))
    if plain_cols:
        steps.append(("num", StandardScaler(), plain_cols))
    if use_cat:
        steps.append(("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=50,
                                           sparse_output=False), CAT_COLS))
    return ColumnTransformer(steps)

def prepare(num_cols, use_cat):
    pre = make_preprocessor(num_cols, use_cat)
    X = {"train": pre.fit_transform(train_df).astype("float32")}          # fit ΜΟΝΟ στο train
    X["val"], X["test"] = pre.transform(val_df).astype("float32"), pre.transform(test_df).astype("float32")
    return pre, X

# Έλεγχος: με όλα τα χαρακτηριστικά πρέπει να βγαίνει ό,τι αποθήκευσε το notebook 03
d = np.load(DATA + "unsw_prepared.npz")
X_full = {s: d[f"X_{s}"] for s in ("train", "val", "test")}
_, X_check = prepare(NUM_COLS, True)
same = all(X_check[s].shape == X_full[s].shape and np.allclose(X_check[s], X_full[s], atol=1e-4) for s in X_full) \
       and all(np.array_equal(y[s]["attack_cat"], d[f"y_{s}_attack_cat"]) for s in X_full)
print("train:", train_df.shape, "| val:", val_df.shape, "| test:", test_df.shape)
print("Ίδια δεδομένα με το notebook 03:", "ΝΑΙ" if same else "ΟΧΙ – σταμάτα εδώ και πες το μου")

## 6.2 Ξαναφέρνουν πλεονασμό τα κατηγορικά;
Ο συντελεστής Pearson δεν ορίζεται για κείμενο. Χρησιμοποιούμε δύο αντίστοιχα μέτρα, και τα δύο από 0 (καμία σχέση) έως 1 (πλήρης εξάρτηση):
- **η (correlation ratio)**: κατηγορικό ↔ αριθμητικό. Πόσο μέρος της διακύμανσης του αριθμητικού εξηγείται από την κατηγορία.
- **Cramér's V**: κατηγορικό ↔ κατηγορικό.

Οι σπάνιες τιμές (< 50 εμφανίσεις) μπαίνουν σε μία ομάδα, όπως κάνει και ο encoder.

In [ ]:
def group_rare(s, min_count=50):
    counts = s.value_counts()
    return s.where(s.map(counts) >= min_count, "(σπάνια)")

def correlation_ratio(cat, x):
    means = x.groupby(cat).agg(["mean", "size"])
    ss_between = (means["size"] * (means["mean"] - x.mean()) ** 2).sum()
    ss_total = ((x - x.mean()) ** 2).sum()
    return float(np.sqrt(ss_between / ss_total)) if ss_total > 0 else 0.0

def cramers_v(a, b):
    table = pd.crosstab(a, b)
    chi2 = chi2_contingency(table, correction=False)[0]
    return float(np.sqrt(chi2 / (table.values.sum() * (min(table.shape) - 1))))

cats = {c: group_rare(train_df[c]) for c in CAT_COLS}
num8 = train_df[FEATS_8].astype("float64").copy()
log8 = [c for c in FEATS_8 if c in LOG_COLS]
num8[log8] = np.log1p(num8[log8])                     # ίδιος μετασχηματισμός με την είσοδο του δικτύου

eta = pd.DataFrame({f: {c: correlation_ratio(cats[c], num8[f]) for c in CAT_COLS} for f in FEATS_8})
print("η: κατηγορικό (γραμμή) ↔ αριθμητικό (στήλη)")
print(eta.round(2).to_string())

print("\nCramér's V μεταξύ κατηγορικών και με την ετικέτα:")
others = {**cats, "label": train_df["label"]}
v_table = pd.DataFrame({b: {a: cramers_v(cats[a], others[b]) for a in CAT_COLS} for b in others})
print(v_table.round(2).to_string())

strong = [(c, f, round(eta.loc[c, f], 2)) for c in CAT_COLS for f in FEATS_8 if eta.loc[c, f] > 0.5]
print(f"\nΖεύγη κατηγορικό–αριθμητικό με η > 0.5: {len(strong)} από {eta.size}")
for c, f, v in sorted(strong, key=lambda t: -t[2]):
    print(f"  {c} ↔ {f}: {v}")

SEQ_CMAP = LinearSegmentedColormap.from_list("seq", ["#f1f0ec", "#2a78d6"])
fig, ax = plt.subplots(figsize=(9.5, 3.2))
sns.heatmap(eta, cmap=SEQ_CMAP, vmin=0, vmax=1, annot=True, fmt=".2f", annot_kws={"size": 10}, linewidths=2, linecolor="white",
            cbar_kws={"label": "η (correlation ratio)"}, ax=ax)
ax.set_title("UNSW-NB15: πόσο εξαρτώνται τα 8 αριθμητικά από τα κατηγορικά")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right"); plt.setp(ax.get_yticklabels(), rotation=0)
plt.tight_layout(); plt.savefig(FIG + "25_unsw_categorical_vs_numeric.png", dpi=200, bbox_inches="tight"); plt.show()

## 6.3 Απλοί κανόνες με τα 8 χαρακτηριστικά
Ίδιος έλεγχος με το notebook 08: αν ένα δέντρο με λίγες ερωτήσεις έφτανε κοντά στο δίκτυο, το δίκτυο δεν θα χρειαζόταν.

In [ ]:
_, X8 = prepare(FEATS_8, False)
names8 = [c for c in LOG_COLS if c in FEATS_8] + [c for c in FEATS_8 if c not in LOG_COLS]     # σειρά στηλών του X8
rules = {"Πάντα «επίθεση»": accuracy_score(y["test"]["label"], np.ones_like(y["test"]["label"]))}
single = {}
for i, f in enumerate(names8):
    m = DecisionTreeClassifier(max_depth=2, random_state=SEED).fit(X8["train"][:, [i]], y["train"]["label"])
    single[f] = accuracy_score(y["test"]["label"], m.predict(X8["test"][:, [i]]))
best = max(single, key=single.get)
rules[f"Καλύτερος κανόνας με ένα χαρακτηριστικό ({best})"] = single[best]
for depth in [3, 5]:
    m = DecisionTreeClassifier(max_depth=depth, random_state=SEED).fit(X8["train"], y["train"]["label"])
    rules[f"Δέντρο απόφασης βάθους {depth} (8 χαρακτηριστικά)"] = accuracy_score(y["test"]["label"], m.predict(X8["test"]))
print(pd.Series(rules, name="accuracy στο test").round(4).to_string())

## 6.4 Δίκτυο και κανόνας απόφασης (ίδια με το notebook 07)
Ίδια αρχιτεκτονική (128-64-32), ίδια βάρη κλάσεων (p = 0.25), ίδιες ρυθμίσεις εκπαίδευσης. Αλλάζουν **μόνο οι είσοδοι**.

In [ ]:
def class_weights(y_cls, n_classes, power):
    counts = np.bincount(y_cls, minlength=n_classes).astype("float64")
    w = (len(y_cls) / (n_classes * np.maximum(counts, 1))) ** power
    return (w / np.average(w, weights=counts)).astype("float32")      # μέσο βάρος = 1

def weighted_sparse_ce(class_w):
    w = keras.ops.convert_to_tensor(class_w, dtype="float32")
    def loss(y_true, y_pred):
        yy = keras.ops.cast(keras.ops.reshape(y_true, (-1,)), "int32")
        return keras.losses.sparse_categorical_crossentropy(yy, y_pred) * keras.ops.take(w, yy)
    return loss

def build_model(n_features, w_attack, w_sev, name, units=(128, 64, 32), dropout=0.3, lr=1e-3):
    inp = keras.Input(shape=(n_features,), name="flow_features")
    x = inp
    for i, u in enumerate(units):
        x = layers.Dense(u, activation="relu", name=f"dense_{i+1}")(x)
        x = layers.BatchNormalization(name=f"bn_{i+1}")(x)
        x = layers.Dropout(dropout, name=f"dropout_{i+1}")(x)
    outs = [layers.Dense(1, activation="sigmoid", name="label")(x),
            layers.Dense(len(ATTACK_CLASSES), activation="softmax", name="attack_cat")(x),
            layers.Dense(len(SEVERITY_CLASSES), activation="softmax", name="severity")(x)]
    model = keras.Model(inp, outs, name=name)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                  loss=["binary_crossentropy", weighted_sparse_ce(w_attack), weighted_sparse_ce(w_sev)],
                  loss_weights=[1.0, 1.0, 0.5])
    return model

def decide(p_label, p_attack, threshold=0.5):
    is_attack = p_label >= threshold
    attack = np.where(is_attack, p_attack[:, 1:].argmax(1) + 1, 0)
    return is_attack.astype(int), attack, SEV_OF_ATTACK[attack]

def evaluate(model, X, split):
    p_label, p_attack, _ = model.predict(X[split], batch_size=1024, verbose=0)
    p_label = p_label.ravel()
    lab, att, sev = decide(p_label, p_attack)
    t = y[split]
    return {"label acc": accuracy_score(t["label"], lab),
            "label ROC-AUC": roc_auc_score(t["label"], p_label),
            "label recall": recall_score(t["label"], lab),
            "label FPR (ψευδείς συναγερμοί)": ((lab == 1) & (t["label"] == 0)).sum() / (t["label"] == 0).sum(),
            "attack_cat acc": accuracy_score(t["attack_cat"], att),
            "attack_cat macro-F1": f1_score(t["attack_cat"], att, average="macro"),
            "attack_cat macro-precision": precision_score(t["attack_cat"], att, average="macro", zero_division=0),
            "attack_cat macro-recall": recall_score(t["attack_cat"], att, average="macro"),
            "severity acc": accuracy_score(t["severity"], sev),
            "severity macro-F1": f1_score(t["severity"], sev, average="macro")}, att

W_ATTACK = class_weights(y["train"]["attack_cat"], len(ATTACK_CLASSES), POWER)
W_SEV = class_weights(y["train"]["severity"], len(SEVERITY_CLASSES), POWER)
print("Βάρη τύπων επίθεσης:", {c: round(float(v), 2) for c, v in zip(ATTACK_CLASSES, W_ATTACK)})

## 6.5 Εκπαίδευση των 4 εκδοχών
| Εκδοχή | Αριθμητικά | Κατηγορικά |
|---|---|---|
| 8 | τα 8 (κανένα ζεύγος με \|r\| > 0.5) | όχι |
| 8 + κατηγορικά | τα 8 | ναι |
| 13 | τα 13 (κατώφλι 0.7· περιέχει `sttl`, `ct_state_ttl`) | όχι |
| 13 + κατηγορικά | τα 13 | ναι |

Για κάθε εκδοχή αποθηκεύονται το δίκτυο και ο preprocessor του (θα χρειαστούν για προβλέψεις σε νέα δεδομένα).

In [ ]:
VARIANTS = {"8": ("8num", FEATS_8, False),
            "8 + κατηγορικά": ("8num_cat", FEATS_8, True),
            "13": ("13num", FEATS_13, False),
            "13 + κατηγορικά": ("13num_cat", FEATS_13, True)}
REF = "39 + κατηγορικά (αρχικό)"

results_val, results_test, preds_test, n_inputs, data = {}, {}, {}, {}, {}
for name, (key, num_cols, use_cat) in VARIANTS.items():
    pre, X = prepare(num_cols, use_cat)
    data[name] = X
    n_inputs[name] = X["train"].shape[1]
    keras.backend.clear_session()
    keras.utils.set_random_seed(SEED)
    model = build_model(X["train"].shape[1], W_ATTACK, W_SEV, name=f"unsw_multitask_{key}")
    history = model.fit(X["train"], targets("train"), validation_data=(X["val"], targets("val")),
                        epochs=60, batch_size=256, verbose=0,
                        callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
                                   keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5)])
    model.save(MODELS + f"unsw_multitask_{key}.keras")
    joblib.dump(pre, MODELS + f"unsw_preprocessor_{key}.joblib")
    results_val[name], _ = evaluate(model, X, "val")
    results_test[name], preds_test[name] = evaluate(model, X, "test")
    print(f"{name}: {n_inputs[name]} είσοδοι, {len(history.history['loss'])} εποχές | "
          f"val label acc = {results_val[name]['label acc']:.4f}, attack_cat macro-F1 = {results_val[name]['attack_cat macro-F1']:.4f}")

# Αναφορά: το δίκτυο του notebook 07 με όλα τα χαρακτηριστικά (δεν ξαναεκπαιδεύεται)
ref_model = keras.models.load_model(MODELS + f"unsw_multitask_p{POWER}.keras", compile=False)
n_inputs[REF] = X_full["train"].shape[1]
results_val[REF], _ = evaluate(ref_model, X_full, "val")
results_test[REF], preds_test[REF] = evaluate(ref_model, X_full, "test")
print(f"{REF}: {n_inputs[REF]} είσοδοι (φορτώθηκε από το notebook 07)")

## 6.6 Σύγκριση
Πρώτα το **validation** (με αυτό διαλέγουμε), μετά το **test** (αυτό αναφέρουμε).

In [ ]:
val_table = pd.DataFrame(results_val).round(4)
test_table = pd.DataFrame(results_test).round(4)
inputs_row = pd.DataFrame([n_inputs], index=["είσοδοι δικτύου"])
print("VALIDATION"); print(pd.concat([inputs_row, val_table]).to_string()); print()
print("TEST"); print(pd.concat([inputs_row, test_table]).to_string())
pd.concat({"validation": val_table, "test": test_table}, axis=1).to_csv(MODELS + "unsw_feature_variants.csv")

best_val = val_table.loc["attack_cat macro-F1", list(VARIANTS)].astype(float).idxmax()
print(f"\nΚαλύτερη μειωμένη εκδοχή στο validation (attack_cat macro-F1): {best_val}")

In [ ]:
metrics = ["label acc", "label ROC-AUC", "attack_cat acc", "attack_cat macro-F1", "severity macro-F1"]
order = ["8", "8 + κατηγορικά", "13", "13 + κατηγορικά", REF]
color_of = {"8": "#2a78d6", "13": "#eb6834", "39": "#8a8a86"}     # χρώμα = πλήθος αριθμητικών· γραμμοσκίαση = + κατηγορικά

fig, ax = plt.subplots(figsize=(12, 4.6))
xpos = np.arange(len(metrics)); width = 0.16
for i, name in enumerate(order):
    with_cat = "κατηγορικά" in name
    ax.bar(xpos + (i - 2) * width, [results_test[name][m] for m in metrics], width * 0.9,
           color=color_of[name.split()[0]], hatch="///" if with_cat else None, edgecolor="white", linewidth=0.8)
ax.set_xticks(xpos); ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.05); ax.grid(axis="y", alpha=0.3); ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Σύστημα Α: επίδοση στο test ανά σύνολο χαρακτηριστικών εισόδου")
ax.legend(handles=[Patch(facecolor=color_of["8"], label="8 αριθμητικά"), Patch(facecolor=color_of["13"], label="13 αριθμητικά"),
                   Patch(facecolor=color_of["39"], label="39 αριθμητικά (αρχικό)"),
                   Patch(facecolor="#c9c8c4", hatch="///", edgecolor="white", label="+ κατηγορικά")],
          ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.1), frameon=False)
plt.tight_layout(); plt.savefig(FIG + "26_unsw_feature_variants.png", dpi=200, bbox_inches="tight"); plt.show()

## 6.7 Ανά τύπο επίθεσης (test)
Recall κάθε τύπου για κάθε εκδοχή: δείχνει **ποιους** τύπους χάνουμε όταν μειώνουμε τα χαρακτηριστικά.

In [ ]:
labels = list(range(len(ATTACK_CLASSES)))
recall_table = pd.DataFrame({name: recall_score(y["test"]["attack_cat"], preds_test[name], labels=labels, average=None, zero_division=0)
                             for name in order}, index=ATTACK_CLASSES)
recall_table.insert(0, "πλήθος", np.bincount(y["test"]["attack_cat"], minlength=len(ATTACK_CLASSES)))
print(recall_table.round(3).to_string())

print(f"\nΑναλυτικά για την εκδοχή «{best_val}»:")
print(classification_report(y["test"]["attack_cat"], preds_test[best_val], labels=labels,
                            target_names=ATTACK_CLASSES, digits=3, zero_division=0))

## 6.8 Random Forest στα ίδια χαρακτηριστικά
Για να δούμε αν το νευρωνικό μένει ισάξιο του κλασικού μοντέλου και με τις μειωμένες εισόδους. (3–8 λεπτά.)

In [ ]:
rf_rows = {}
for name in VARIANTS:
    X = data[name]
    make_rf = lambda: RandomForestClassifier(n_estimators=200, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED)
    pl = make_rf().fit(X["train"], y["train"]["label"]).predict_proba(X["test"])[:, 1]
    pa = make_rf().fit(X["train"], y["train"]["attack_cat"]).predict(X["test"])
    rf_rows[name] = {"RF label acc": accuracy_score(y["test"]["label"], pl >= 0.5),
                     "ΝΔ label acc": results_test[name]["label acc"],
                     "RF ROC-AUC": roc_auc_score(y["test"]["label"], pl),
                     "ΝΔ ROC-AUC": results_test[name]["label ROC-AUC"],
                     "RF attack_cat acc": accuracy_score(y["test"]["attack_cat"], pa),
                     "ΝΔ attack_cat acc": results_test[name]["attack_cat acc"],
                     "RF attack_cat macro-F1": f1_score(y["test"]["attack_cat"], pa, average="macro"),
                     "ΝΔ attack_cat macro-F1": results_test[name]["attack_cat macro-F1"]}
    print(name, "έτοιμο")
rf_table = pd.DataFrame(rf_rows).T.round(4)
rf_table.to_csv(MODELS + "unsw_feature_variants_rf.csv")
print(rf_table.to_string())

## Τι κρατάμε
- Αν η εκδοχή **8** (ή **8 + κατηγορικά**) είναι κοντά στο αρχικό, συνεχίζουμε με αυτήν: ίδιο αποτέλεσμα με πολύ λιγότερη και μη πλεονάζουσα πληροφορία.
- Αν τα κατηγορικά έχουν μεγάλο η με τα αριθμητικά (6.2) και δεν βελτιώνουν αισθητά το validation, τα αφήνουμε έξω.
- Το **13** είναι μόνο σημείο σύγκρισης: έχει ακόμα 12 ζεύγη με |r| > 0.5 και τα χαρακτηριστικά TTL.